# Feogasy — Test VoxCPM2 sur Google Colab

**Projet :** Feogasy — Binôme TTS (Mahefa + David)
**Objectif :** Tester l'installation et l'inférence VoxCPM2 avec GPU gratuit, sans impacter le stockage local.

**Avant de commencer :**
1. Menu `Exécution` → `Modifier le type d'exécution` → sélectionner **GPU** (T4)
2. Exécuter les cellules dans l'ordre (Shift+Entrée)

**Note :** utiliser uniquement du texte et des voix dont l'usage est autorisé (règle du projet Feogasy).

## 1. Vérification du GPU

In [ ]:
import torch
print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun — vérifier le type d'exécution")

## 2. (Optionnel mais recommandé) Monter Google Drive pour mettre en cache le modèle

Évite de retélécharger les ~4-5 Go de poids à chaque nouvelle session Colab. Le modèle sera stocké dans ton Google Drive et réutilisé automatiquement la prochaine fois.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/feogasy_hf_cache', exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/feogasy_hf_cache'
print("Cache Hugging Face configuré sur Google Drive :", os.environ['HF_HOME'])

## 3. Installation de VoxCPM2

Colab fournit déjà PyTorch avec CUDA préinstallé, donc pas besoin de gérer les versions CPU/GPU manuellement.

In [ ]:
!pip install voxcpm soundfile -q

## 4. Vérification de l'import

In [ ]:
import voxcpm
print("voxcpm importé avec succès — version:", voxcpm.__version__ if hasattr(voxcpm, '__version__') else "inconnue")

## 5. Chargement du modèle VoxCPM2

Premier lancement : téléchargement des poids (~4-5 Go) depuis Hugging Face. Peut prendre quelques minutes selon la charge du Hub.
Si le cache Drive (étape 2) est activé, les lancements suivants seront quasi instantanés.

In [ ]:
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
print("Modèle VoxCPM2 chargé avec succès.")

## 6. Test d'inférence — synthèse vocale simple

Utiliser uniquement un texte de test autorisé.

In [ ]:
import soundfile as sf
import time

texte_test = "Ceci est un test de synthese vocale avec VoxCPM2 pour le projet Feogasy."

debut = time.time()
wav = model.generate(
    text=texte_test,
    cfg_value=2.0,
    inference_timesteps=10,
    seed=42,
)
duree_generation = time.time() - debut

sf.write("demo.wav", wav, model.tts_model.sample_rate)
duree_audio = len(wav) / model.tts_model.sample_rate
rtf = duree_generation / duree_audio

print(f"Audio genere : demo.wav")
print(f"Duree de generation : {duree_generation:.2f}s")
print(f"Duree de l'audio : {duree_audio:.2f}s")
print(f"RTF (Real-Time Factor) : {rtf:.3f}  (objectif projet : < 0.5)")

## 7. Écouter le résultat

In [ ]:
from IPython.display import Audio
Audio("demo.wav")

## 8. (Optionnel) Test Voice Design — voix générée à partir d'une description

Pas besoin d'audio de référence. La description de voix se met entre parenthèses au début du texte.

In [ ]:
wav_design = model.generate(
    text="(voix masculine, calme et posee) Bonjour, ceci est un test de conception de voix.",
    cfg_value=2.0,
    inference_timesteps=10,
    seed=42,
)
sf.write("voice_design.wav", wav_design, model.tts_model.sample_rate)
Audio("voice_design.wav")

## 9. Télécharger les fichiers audio générés sur ton PC (optionnel)

In [ ]:
from google.colab import files
files.download("demo.wav")
# files.download("voice_design.wav")  # decommenter si besoin

## 10. Résumé pour `docs/installation.md`

Copier les résultats ci-dessus (RTF, succès du chargement, GPU utilisé) dans le rapport d'installation du projet Feogasy.

**État :**
- [ ] GPU Colab confirmé : ___
- [ ] Installation réussie : oui / non
- [ ] Chargement du modèle réussi : oui / non
- [ ] RTF mesuré : ___
- [ ] Audio généré et écouté avec succès : oui / non